# Week 2 - Mini Project: Plant Leaf Disease Classification System

## 1. Project Overview & Deliverables
This notebook documents the end-to-end development of the **Plant Leaf Disease Diagnostic Model** using ResNet50 transfer learning.
The model trained and evaluated here is saved to `models/plant_disease_resnet50.keras` and forms the inference engine of the Flask web application in `app.py`.

### System Requirements
* Input: RGB image of plant leaf ($224 \times 224 \times 3$).
* Backbone: Pretrained ResNet50 with fine-tuned residual blocks.
* Output: 5-class probability distribution with top-3 candidate rankings.
* Preprocessing: ImageNet BGR zero-centering (`caffe` style via `resnet50.preprocess_input`).


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models, optimizers

import sys
sys.path.append("..")
import model_utils

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


In [ ]:
DATA_DIR = os.path.join("data", "plant_village_5_classes")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen = val_datagen.flow_from_directory(VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen = test_datagen.flow_from_directory(TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

print(f"Classes: {train_gen.class_indices}")


In [ ]:
# Build or Load Model
MODEL_PATH = os.path.join("models", "plant_disease_resnet50.keras")

if os.path.exists(MODEL_PATH):
    print(f"Loading existing trained model from {MODEL_PATH}...")
    model = tf.keras.models.load_model(MODEL_PATH)
else:
    print("Building and training ResNet50 transfer learning model...")
    model = model_utils.build_resnet50_model(num_classes=5, freeze_backbone=True)
    model.compile(optimizer=optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_gen, validation_data=val_gen, epochs=12, verbose=1)
    
    # Fine-tune conv5_block3
    for layer in model.layers:
        if 'conv5_block3' in layer.name:
            layer.trainable = True
    model.compile(optimizer=optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_gen, validation_data=val_gen, epochs=6, verbose=1)
    
    model.save(MODEL_PATH)
    print(f"Saved trained model to {MODEL_PATH}")


In [ ]:
# Full Test Set Evaluation
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print("=" * 60)
print(f"PLANT DISEASE DETECTOR TEST ACCURACY: {test_acc * 100:.2f}%")
print(f"PLANT DISEASE DETECTOR TEST LOSS:     {test_loss:.4f}")
print("=" * 60)

preds = model.predict(test_gen, verbose=0)
pred_classes = np.argmax(preds, axis=1)
true_classes = test_gen.classes
class_names = list(train_gen.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title(f"Plant Disease Detector Confusion Matrix (Accuracy: {test_acc*100:.2f}%)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Per-Class Classification Report:")
print(classification_report(true_classes, pred_classes, target_names=class_names, digits=4))


## 2. Interactive Single-Image Inference Demonstration
Simulating the exact preprocessing and inference pipeline invoked by `app.py` via `model_utils.predict_image()`.


In [ ]:
# Select a test sample file
sample_test_file = os.path.join(TEST_DIR, "Tomato___healthy", os.listdir(os.path.join(TEST_DIR, "Tomato___healthy"))[0])

with open(sample_test_file, "rb") as f:
    sample_bytes = f.read()

# Execute inference via model_utils
result = model_utils.predict_image(sample_bytes, model=model)

print("INFERENCE RESULT:")
print(f"Predicted Class:      {result['predicted_class']}")
print(f"Human-Readable Label: {result['predicted_label']}")
print(f"Model Confidence:     {result['confidence_percentage']}")
print("\nTop-3 Pathology Candidates:")
for item in result['top_predictions']:
    print(f"  - {item['label']:<30}: {item['percentage']}")


## 3. Deployment Summary
* **Model Artifact**: Stored at `Week-2/models/plant_disease_resnet50.keras` (size: ~98 MB).
* **Class Mapping**: Stored at `Week-2/models/class_indices.json`.
* **Web Interface**: Launch the Flask application with `python app.py` and navigate to `http://localhost:5000` to upload and analyze plant leaves in real time.
